In [1]:
from pathlib import Path

import igl
import meshlib.mrmeshnumpy as mrmeshnumpy
import meshlib.mrmeshpy as mrmeshpy
import numpy as np
import pyvista as pv

In [7]:
displaced_mesh = pv.read("displaced_mesh.vtk")
plotter = pv.Plotter()
plotter.add_mesh(displaced_mesh, style="wireframe", color="grey")
plotter.show()

Widget(value='<iframe src="http://localhost:43603/index.html?ui=P_0x7fb6438d25d0_5&reconnect=auto" class="pyvi…

In [3]:
def map_path_to_circle(center, radius, orientation, start_angle, path):
    x_coordinates = np.zeros(len(path))
    y_coordinates = np.zeros(len(path))

    for i, _ in enumerate(path):
        angle = i / len(path) * 2 * np.pi * orientation + start_angle
        x_coordinates[i] = center[0] + radius * np.cos(angle)
        y_coordinates[i] = center[1] + radius * np.sin(angle)

    return x_coordinates, y_coordinates

In [4]:
mesh = pv.read("dummy_mesh_one_hole.vtu")
outer_radius = 5
inner_radius = 2
center = (0, 0)
start_angle = 0

In [5]:
vertices = np.array(mesh.points)
simplices = np.array(mesh.cells.reshape(-1, 4)[:, 1:4])
boundary_loops = igl.boundary_loop_all(simplices)
inner_boundary = np.array(boundary_loops[0])
outer_boundary = np.array(boundary_loops[1])

In [6]:
meshlib_mesh = mrmeshnumpy.meshFromFacesVerts(simplices, vertices)
hole_edges = meshlib_mesh.topology.findHoleRepresentiveEdges()
inner_boundary = hole_edges[0]
params = mrmeshpy.FillHoleParams()
new_faces_bitset = mrmeshpy.FaceBitSet()
params.outNewFaces = new_faces_bitset
mrmeshpy.fillHole(meshlib_mesh, inner_boundary, params)
extended_vertices = mrmeshnumpy.getNumpyVerts(meshlib_mesh)
extended_simplices = mrmeshnumpy.getNumpyFaces(meshlib_mesh.topology)
extended_mesh = pv.PolyData.from_regular_faces(extended_vertices, extended_simplices)

In [7]:
x_coordinates_outer, y_coordinates_outer = map_path_to_circle(
    center,
    outer_radius,
    1,
    start_angle,
    outer_boundary,
)
boundary_coordinates = np.column_stack((x_coordinates_outer, y_coordinates_outer))
harmonic_map = igl.harmonic(
    extended_vertices, extended_simplices, outer_boundary, boundary_coordinates, k=1
)
z_coordinates = np.zeros(len(harmonic_map))
harmonic_map = np.column_stack((harmonic_map, z_coordinates))

In [8]:
plotter = pv.Plotter()
uac_mesh = pv.PolyData.from_regular_faces(harmonic_map, extended_simplices)
plotter.add_mesh(uac_mesh, style="wireframe", color="grey")
plotter.show()

Widget(value='<iframe src="http://localhost:41081/index.html?ui=P_0x7f040a285450_1&reconnect=auto" class="pyvi…

In [9]:
import meshio

output_mesh = meshio.Mesh(
    points=harmonic_map[:, :2],
    cells=[("triangle", simplices)],
)

output_mesh.write("uac_mesh_1.xdmf")